# Eager broadcasting and message ordering

## Question 1

Paste your code for the eager broadcasting solution below and explain why your
solution solves the high-drop-rate problem (the failing unit test).

In [ ]:
import uuid
from dataclasses import dataclass, field
from typing import Any

from dslabs.protocols import Message, Scheduler, Transport


@dataclass
class NodeEagerBroadcast:
    """Multi-leader replication where every node rebroadcasts what it receives."""

    node_id: str
    peers: list[str]
    transport: Transport
    scheduler: Scheduler
    store: dict[str, Any] = field(default_factory=dict)
    log: list[tuple[str, Any]] = field(default_factory=list)
    # Ids of the messages this node has already applied and forwarded
    seen: set[str] = field(default_factory=set)

    # Client-facing API
    def client_put(self, key: str, value: Any) -> None:
        self.receive_and_replicate(key, value, str(uuid.uuid4()))

    def client_get(self, key: str) -> Any:
        return self.store.get(key)

    # Node internals
    def receive_and_replicate(self, key: str, value: Any, msg_id: str) -> None:
        if msg_id in self.seen:
            return
        self.seen.add(msg_id)
        self.receive(key, value)
        for peer in self.peers:
            if peer != self.node_id:
                self.transport.send(peer, {"type": "replicate", "key": key, "value": value, "id": msg_id})

    def receive(self, key: str, value: Any) -> None:
        # Received messages are delivered immediately
        self.deliver(key, value)

    def deliver(self, key: str, value: Any) -> None:
        """Apply a message to this node: update the store, append to the log."""
        self.store[key] = value
        self.log.append((key, value))

    # Network handler
    def on_message(self, msg: Message) -> None:
        if msg["type"] == "replicate":
            self.receive_and_replicate(msg["key"], msg["value"], msg["id"])
        else:
            raise ValueError(f"Unknown message type in {msg!r}")

    # What the trace and diagrams show as this node's state
    def brief_state(self) -> dict[str, Any]:
        return dict(self.store)

This is my implementation for the eager broadcasting node. I copied the multi leader node and changed these things:

- Added a `seen` set that holds the ids of the messages this node already handled
- `client_put` now makes a `uuid4` for the write before replicating it, so the write has an id that travels with it
- `receive_and_replicate` takes that id and returns straight away if it is already in `seen`. If it is new it adds it, applies the write, and sends it on to every peer
- The replicate message carries the id now, and `on_message` passes it through

Why this fixes the drop test:

In the multi leader node a write is broadcast once, by the node the client wrote to. So every other node gets exactly one chance at it, and with 50% drops about half of them miss any given write. Nothing ever resends it, so a node that missed the last write just keeps an old value and the nodes end up disagreeing. Running it I get `[10, 11, 5, 9]`, four different values.

With eager broadcasting every node that sees the write for the first time also forwards it to all of its peers. So the write gets to a node through lots of different paths instead of only the direct one from the sender. For a node to miss it now the direct copy has to be dropped *and* every forwarded copy from all the other nodes, which with 12 nodes basically never happens. All 12 end on `11`.

The cost is traffic. Multi leader sends 132 messages for this run, eager broadcasting sends 1584. That redundancy is the whole point, it is what survives the drops.

The `seen` set is what stops it running forever. Without it n1 forwards to n2, n2 forwards back to n1, and the same write bounces around the cluster for ever. Because each write has its own uuid, a node forwards a given id once and ignores every copy after that.

One thing this does not do is order anything. Nothing about flooding says which write happened first, so on the fast write test the 6 nodes still end up with 6 different logs and 6 different values. That is question 2.

## Question 2

Paste your code for delivering messages in the correct order. Explain how your
solution works to solve the high-send-rate problem (the failing unit test).

In [ ]:
import uuid
from dataclasses import dataclass, field
from typing import Any

from dslabs.protocols import Message, Scheduler, Transport


@dataclass
class NodeTotalOrder:
    """Replication through a sequencer, so every node delivers in the same order."""

    node_id: str
    peers: list[str]
    transport: Transport
    scheduler: Scheduler
    store: dict[str, Any] = field(default_factory=dict)
    log: list[tuple[str, Any]] = field(default_factory=list)
    # Requests already forwarded, so a flooded request is not forwarded twice
    seen_requests: set[str] = field(default_factory=set)
    # Numbered writes this node has received, by sequence number
    history: dict[int, tuple[str, Any, str]] = field(default_factory=dict)
    # The next sequence number this node will apply
    next_delivery: int = 0
    # This node's own writes that have not come back yet, by request id
    pending: dict[str, tuple[str, Any]] = field(default_factory=dict)
    # Sequencer only: the next number to hand out, what it numbered, and who
    # still owes an acknowledgement for each number
    next_seq: int = 0
    numbered: dict[str, int] = field(default_factory=dict)
    unacked: dict[int, set[str]] = field(default_factory=dict)
    retry_ms: int = 300
    retry_armed: bool = False

    @property
    def sequencer_id(self) -> str:
        return self.peers[0]

    # Client-facing API
    def client_put(self, key: str, value: Any) -> None:
        msg_id = str(uuid.uuid4())
        self.pending[msg_id] = (key, value)
        if self.node_id == self.sequencer_id:
            self.receive_request(key, value, msg_id)
        else:
            self.broadcast({"type": "request", "key": key, "value": value, "id": msg_id})
        self.arm_retry()

    def client_get(self, key: str) -> Any:
        return self.store.get(key)

    # Node internals
    def receive_request(self, key: str, value: Any, msg_id: str) -> None:
        """Sequencer only: give a write its number, unless it already has one."""
        if msg_id in self.numbered:
            return
        seq = self.next_seq
        self.next_seq += 1
        self.numbered[msg_id] = seq
        self.unacked[seq] = {p for p in self.peers if p != self.node_id}
        self.receive_ordered(key, value, seq, msg_id)
        for peer in self.peers:
            if peer != self.node_id:
                self.send_ordered(peer, seq)
        self.arm_retry()

    def send_ordered(self, to: str, seq: int) -> None:
        key, value, msg_id = self.history[seq]
        self.transport.send(to, {"type": "ordered", "key": key, "value": value, "seq": seq, "id": msg_id})

    def receive_ordered(self, key: str, value: Any, seq: int, msg_id: str) -> None:
        if seq in self.history:
            return
        self.history[seq] = (key, value, msg_id)
        while self.next_delivery in self.history:
            k, v, applied_id = self.history[self.next_delivery]
            self.deliver(k, v)
            self.pending.pop(applied_id, None)
            self.next_delivery += 1

    def broadcast(self, msg: Message) -> None:
        for peer in self.peers:
            if peer != self.node_id:
                self.transport.send(peer, msg)

    def deliver(self, key: str, value: Any) -> None:
        """Apply a message to this node: update the store, append to the log."""
        self.store[key] = value
        self.log.append((key, value))

    # Retries: resend anything still outstanding until it is answered
    def arm_retry(self) -> None:
        if not self.retry_armed and (self.pending or self.unacked):
            self.retry_armed = True
            self.scheduler.call_later(self.retry_ms, self.retry)

    def retry(self) -> None:
        self.retry_armed = False
        for msg_id, (key, value) in list(self.pending.items()):
            # Straight at the sequencer this time; the first broadcast did not do it
            if self.node_id == self.sequencer_id:
                self.receive_request(key, value, msg_id)
            else:
                self.transport.send(
                    self.sequencer_id, {"type": "request", "key": key, "value": value, "id": msg_id}
                )
        for seq, waiting in list(self.unacked.items()):
            for peer in self.peers:
                if peer in waiting:
                    self.send_ordered(peer, seq)
        self.arm_retry()

    # Network handler
    def on_message(self, msg: Message) -> None:
        if msg["type"] == "request":
            if self.node_id == self.sequencer_id:
                self.receive_request(msg["key"], msg["value"], msg["id"])
            elif msg["id"] not in self.seen_requests:
                self.seen_requests.add(msg["id"])
                self.broadcast(msg)
        elif msg["type"] == "ordered":
            # Acknowledge every copy, including one we already have: if an
            # acknowledgement is lost the sequencer will send the write again.
            self.transport.send(msg["from"], {"type": "ack", "seq": msg["seq"]})
            self.receive_ordered(msg["key"], msg["value"], msg["seq"], msg["id"])
        elif msg["type"] == "ack":
            waiting = self.unacked.get(msg["seq"])
            if waiting is not None:
                waiting.discard(msg["from"])
                if not waiting:
                    del self.unacked[msg["seq"]]
        else:
            raise ValueError(f"Unknown message type in {msg!r}")

    # What the trace and diagrams show as this node's state
    def brief_state(self) -> dict[str, Any]:
        return dict(self.store)

This is my implementation for delivering messages in the correct order. How it works:

- One node is the sequencer, I just use the first node in `peers`
- When a client writes to any node, that node makes a uuid for the write and gets it to the sequencer as a `request`
- The sequencer gives the write the next number from `next_seq` and sends the numbered write to everyone as an `ordered` message
- Every node keeps `history`, which maps a number to the write, and `next_delivery`, which is the next number it is allowed to apply
- When an ordered message arrives it goes into `history` first, then the node applies everything it can starting from `next_delivery`. So if number 5 turns up and the node has only applied up to 3, then 5 just sits in history until 4 arrives

Why this fixes the fast write test:

In that test the writes go out 2ms apart but the network takes 50 to 200ms, so the replication messages overtake each other and every node applies them in whatever order they land in. Last write wins, so the nodes end on different values. With eager broadcasting I get 6 different logs across 6 nodes and values `[19, 20, 21, 23, 24, 27]`.

Sending everything through the sequencer means there is one order, and the number is what carries that order inside the message instead of leaving it up to arrival time. Since a node refuses to apply a number before it has applied all the numbers below it, the arrival order stops mattering. Every node applies 0, 1, 2, 3 and so on, so every node's `log` comes out identical and the `store` follows from that. 1 distinct log across 6 nodes, all on `27`.

Note: my first version used eager broadcasting for the ordered messages as well and that was not enough. 4 out of 40 seeds hung, with every follower missing number 0 and sitting on 1 to 11 in history for ever. Once you block on a gap, one lost message stops that node applying anything again, so it is much worse than losing a write. Then I tried having a node ask for numbers it could see it was missing, which fixed nearly all of it, but one seed still failed with a node missing the *last* number. It had nothing after it so it had no way to know it was missing anything at all.

So the ordered messages are acknowledged instead. The sequencer keeps `unacked` for each number and resends to whoever has not answered, and a node keeps `pending` for its own write and nags the sequencer until it sees it applied. Both stop as soon as they are answered so the cluster still goes idle at the end.

The requests are still broadcast eagerly rather than sent straight to the sequencer. That is because the sequencer numbers writes in the order it hears about them, so a slow request gets numbered behind a write the client made later and then the wrong value ends up last. When I sent requests directly and just retried them, 4 out of 60 seeds finished on write 10 instead of 11. Broadcasting gets them there in about one hop.

## Checking it

The `log` and `store` on every node come out the same, under fast writes and under heavy drops.

In [ ]:
from dslabs.nodes import NodeMultiLeader, NodeEagerBroadcast, NodeTotalOrder
from dslabs.simulations.sim_send_many import SimSendMany

drops = dict(num_nodes=12, num_messages=12, interval_ms=1000, drop_prob=0.5, seed=12345)
fast = dict(num_nodes=6, num_messages=30, interval_ms=2, drop_prob=0.0, seed=42)

for name, kw in [("50% drops", drops), ("fast writes", fast)]:
    for node_class in (NodeMultiLeader, NodeEagerBroadcast, NodeTotalOrder):
        sim = SimSendMany(node_class, **kw)
        values = sim.run()
        logs = {tuple(n.log) for n in sim.nodes.values()}
        stores = {tuple(sorted(n.store.items())) for n in sim.nodes.values()}
        print(f"{name:12} {node_class.__name__:20} "
              f"logs equal: {str(len(logs) == 1):5} stores equal: {str(len(stores) == 1):5} "
              f"values: {sorted(set(values.values()), key=str)}")
    print()